# rPPG — 9. tur eğitimi: yeni eğitim teknikleri (G) ve standart protokol (S1, S2) (Kaggle GPU)

| Deney | Model | Başlangıç | Soru |
|---|---|---|---|
| **G** | MMRPhys | 7. tur E'nin son epoch'u | Yeni teknikler birlikte hatayı düşürür mü? Spektral kayıp (0.2), EMA (0.999), nabız dengeli örnekleme (3×), geniş hız artırması (0.7–1.5×), **yüz kutusu kayması** (±%10; uygulamadaki pico/YuNet kutusu eğitimdeki Haar kutusundan farklı) |
| **S1** | MMRPhys | rastgele (sıfırdan) | **Standart protokol: yalnızca PURE ile eğit, UBFC-rPPG'nin 42 kişisinin hepsinde test et.** Literatürle doğrudan karşılaştırılabilir rakam |
| **S2** | MMRPhys | rastgele (sıfırdan) | Standart protokol, ters yön: yalnızca UBFC-rPPG ile eğit, PURE'un tamamında test et |

**Neden:** 8. turda UBFC-Phys eklemek fark yaratmadı (E'ye göre hiçbir test anlamlı değil). Uygulama içinde model
farkları ölçülemedi; uygulamadaki hata (3–4 BPM) çevrimdışı hatanın (1.1–1.6) iki katı. G bu farkı hedefler.
S1/S2 olmadan "literatürdeki en iyi" denemez: bizim test kümelerimiz kendi tanımımız.

**Sıra önerisi:** önce **S1** (yalnızca UBFC + PURE indirir, ~3-4 saat), sonra **G** (tam veri, ~11 saat), sonra S2.

**Model seçimi:** son epoch (last.pth); best.pth ek satır.

**Kaggle:**
1. **Accelerator: GPU T4 x2**, **Internet: on**.
2. **Add-ons → Secrets:** `HF_TOKEN`, `HF_WRITE_TOKEN`.
3. Aşağıda `DENEY` seçin → **Save Version → Save & Run All**.

Oturum biterse ya da düşerse **aynı deneyle tekrar çalıştırın**: sonuçlar 30 dakikada bir Hugging Face'teki gizli
`rppg-cikti/tur9_<deney>/` klasörüne yedeklenir ve eğitim `last.pth`'den devam eder.

**Disk:** hazırlama 97 GB'ta kendiliğinden durur (önceki "Read-only file system" ölümleri). DLCN Kaggle girdisi eklenmez.


In [ ]:
# 0) Deney
DENEY = "S1"

TUR7E = "results/derin_ogrenme/kaggle_tur7_E/last.pth"
E_PAYLAR = "--source-weights mcd=0.70,vipl=0.15,dlcn=0.08,cohface=0.03,pure=0.02,ubfc=0.02"   # 7E ile aynı
DENEYLER = {   # ad, dl_train.py argümanları
    "G": ("MMRPhys 7E'den + yeni teknikler (9. tur)",
          f"--arch mmrphys --init {TUR7E} --lr 2e-4 --val-max 60 {E_PAYLAR} "
          "--spec-loss 0.2 --ema 0.999 --hr-balance 3 --speed 0.7 1.5 --box-jitter 0.1"),
    # S1/S2: yalnızca bir veri setinden örneklenir (diğerlerinin payı 0); literatürdeki gibi az epoch
    "S1": ("MMRPhys sıfırdan, yalnızca PURE (standart protokol)",
           "--arch mmrphys --scratch --lr 1e-3 --val-max 40 --source-weights pure=1"),
    "S2": ("MMRPhys sıfırdan, yalnızca UBFC-rPPG (standart protokol)",
           "--arch mmrphys --scratch --lr 1e-3 --val-max 40 --source-weights ubfc=1"),
}
AD, TRAIN_ARGS = DENEYLER[DENEY]
RUN = f"tur9_{DENEY}"
STD = DENEY.startswith("S")          # standart protokol: yalnızca UBFC + PURE hazırlanır
EPOCHS, STEPS = (30, 500) if STD else (100, 500)
EK_VERI = not STD   # DLCN + COHFACE eğitimde (E ile aynı)
# True: eğitimi biten deneyin (HF yedeğinde last.pth son epoch) yalnızca değerlendirmesi, GPU'suz (Accelerator: None).
# 9. tur S1 eğitimi bitti ama değerlendirme dosya adı hatasıyla çöktü (düzeltildi); böyle durumlar için.
SADECE_DEGERLENDIRME = False


In [ ]:
# 1) Ortam, kod (rppg, rPPG-Toolbox, MMRPhys sabit sürüm), anahtarlar
import os, subprocess, shutil, sys, time, glob
from datetime import datetime, timedelta

T0 = datetime.now()
# Hugging Face: xet parça önbelleği diske ~10 GB yazabiliyor, ilerleme çubukları günlükleri şişiriyor
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
KAGGLE = os.path.isdir("/kaggle/working")
WORK = "/kaggle/working" if KAGGLE else "/content"
print("ortam:", "Kaggle" if KAGGLE else "Colab")
if shutil.which("nvidia-smi"):   # GPU'suz oturumda komut yok (yalnızca değerlendirme)
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert shutil.which("nvidia-smi") or SADECE_DEGERLENDIRME, ("GPU yok: Settings -> Accelerator: GPU T4 x2" if KAGGLE else
                                    "GPU yok: Çalışma zamanı -> Çalışma zamanı türünü değiştir -> T4 GPU")
assert subprocess.run(["curl", "-sI", "https://github.com"], capture_output=True).returncode == 0, \
    "İnternet kapalı: Settings -> Turn on internet"

REPO = os.path.join(WORK, "rppg")
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Nisagwn/rppg.git", REPO], check=True)
os.chdir(REPO)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
TB = os.path.join(REPO, "external", "rPPG-Toolbox")
if not os.path.exists(TB):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ubicomplab/rPPG-Toolbox.git", TB], check=True)
sys.path.insert(0, os.path.join(REPO, "scripts"))
from dl_common import MMRPHYS, MMRPHYS_COMMIT
if not os.path.exists(MMRPHYS):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/PhysiologicAILab/MMRPhys.git", MMRPHYS], check=True)
    head = subprocess.run(["git", "-C", MMRPHYS, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    if head != MMRPHYS_COMMIT:   # depo güncellendiyse yerelde denenen sürüme dön
        subprocess.run(["git", "-C", MMRPHYS, "fetch", "--depth", "1", "origin", MMRPHYS_COMMIT], check=True)
        subprocess.run(["git", "-C", MMRPHYS, "checkout", MMRPHYS_COMMIT], check=True)
print("MMRPhys", subprocess.run(["git", "-C", MMRPHYS, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout)

def secret(name):
    if KAGGLE:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    from google.colab import userdata
    return userdata.get(name)

for name in ("HF_TOKEN", "HF_WRITE_TOKEN"):
    try:
        os.environ[name] = secret(name)
        print(f"{name} var")
    except Exception:
        raise SystemExit(f"{name} yok: " + ("Add-ons -> Secrets -> kutusunu işaretleyip defteri yeniden başlatın" if KAGGLE
                         else "soldaki 🔑 Gizli anahtarlar -> ekleyip 'Not defteri erişimi'ni açın"))

import psutil
from huggingface_hub import HfApi, snapshot_download
API = HfApi(token=os.environ["HF_WRITE_TOKEN"])
OUT_REPO = f"{API.whoami()['name']}/rppg-cikti"
DATA, RAW = "/tmp/dl", "/tmp/dl_raw"
OUT = os.path.join(WORK, "cikti")
FREE0 = shutil.disk_usage("/tmp").free / 1e9   # Colab'da df gerçek; Kaggle'da değil (bkz. disk notu)
LOGS = os.path.join(OUT, "gunlukler")
for d in (DATA, RAW, OUT, LOGS):
    os.makedirs(d, exist_ok=True)
TUR5 = os.path.join(REPO, "results/derin_ogrenme/kaggle_tur5/last.pth")   # hizalama referansı, karşılaştırma
TUR3 = os.path.join(REPO, "results/derin_ogrenme/kaggle_tur3/last.pth")   # uygulamadaki model
TUR7B = os.path.join(REPO, "results/derin_ogrenme/kaggle_tur7_B/best.pth")  # MMRPhys sıfırdan
TUR7E = os.path.join(REPO, TUR7E)  # 8. turun başlangıcı ve karşılaştırma tabanı
TRAIN_UNTIL = T0 + timedelta(hours=12 - 1.6)          # değerlendirme ve son yedek için pay
print(f"deney {DENEY}: {AD} | {TRAIN_ARGS} | disk /tmp boş {FREE0:.0f} GB"
      f" | RAM {psutil.virtual_memory().total / 1e9:.0f} GB | eğitim en geç {TRAIN_UNTIL:%H:%M}")

In [ ]:
# 2) Önceki oturumdan devam: Hugging Face'teki yedek (last.pth, best.pth, gecmis.json)
import threading
_bk = None
def _upload(note):
    try:
        API.create_repo(OUT_REPO, repo_type="dataset", private=True, exist_ok=True)
        API.upload_folder(folder_path=OUT, repo_id=OUT_REPO, repo_type="dataset", path_in_repo=RUN,
                          commit_message=f"{RUN}: {note}")
        print(f"yedeklendi ({note}): huggingface.co/datasets/{OUT_REPO}/tree/main/{RUN}", flush=True)
    except Exception as e:
        print("yedekleme hatası:", repr(e), flush=True)

def backup(note, wait=0):
    """Arka planda yükler: ağ takılsa da izleme döngüsü durmaz (7. tur B'de 22:31'den sonra yedek gelmedi).
    wait > 0: en çok o kadar saniye bekler (son yedekler)."""
    global _bk
    if _bk is not None and _bk.is_alive():
        print("önceki yedekleme sürüyor, bu tur atlandı", flush=True)
    else:
        _bk = threading.Thread(target=_upload, args=(note,), daemon=True)
        _bk.start()
    if wait:
        _bk.join(wait)

try:
    snapshot_download(OUT_REPO, repo_type="dataset", local_dir="/tmp/onceki", token=API.token,
                      allow_patterns=[f"{RUN}/last.pth", f"{RUN}/best.pth", f"{RUN}/gecmis.json"])
    for f in glob.glob(f"/tmp/onceki/{RUN}/*"):
        shutil.copy(f, OUT)
    print("önceki oturum:", sorted(os.listdir(OUT)))
except Exception as e:
    print("önceki oturum yok, baştan:", type(e).__name__)

In [ ]:
# 3) VIPL-HR, DLCN, COHFACE, UBFC-Phys (Hugging Face'teki rppg-veri, rppg_hazirla.ipynb ile hazırlanmış) + model tabanlı hizalama.
#    DLCN Kaggle girdisi (43 GB) bu oturumda okunmaz: çöken üç oturumun ortak yanı DLCN'in okunmasıydı.
PY = sys.executable
env = dict(os.environ, PYTHONUNBUFFERED="1")
def run(cmd, name):
    t = time.time()
    r = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True, encoding="utf-8", errors="replace", env=env)
    open(os.path.join(LOGS, name + ".log"), "w", encoding="utf-8").write(r.stdout + "\n" + r.stderr)
    print(f"[{name}] çıkış {r.returncode}, {time.time() - t:.0f} s\n", r.stdout[-2500:], r.stderr[-1500:], flush=True)
    return r

if not STD:
    HF_SUBS = ["vipl", "test_vipl", "test_mmpd", "test_dlcn", "test_cohface", "ubfcphys", "test_ubfcphys"] + (["dlcn", "cohface"] if EK_VERI else [])
    run([PY, "-u", "scripts/dl_hf_sync.py", "--data", DATA, "--subs", *HF_SUBS, "--download", "--ignore", "*_source2.*"],
        "veri_hf")
    run([PY, "-u", "scripts/dl_sync_check.py", "--write", "--data", DATA,
         "--subs", "vipl", "ubfcphys", *(["dlcn", "cohface"] if EK_VERI else []), "--model", TUR5], "hizala")

In [ ]:
# 4) Diğer veri (arka planda, indir-işle-sil): MCD, test kümeleri, UBFC, PURE.
#    Disk: Kaggle'ın yazılabilir alanı df'nin gösterdiğinden (~1 TB) çok küçük. 6. tur ve 7. tur B (iki kez) oturumun
#    yazdığı veri ~90-100 GB'a varınca "Read-only file system" ile durdu (RAM 30 GB boştu). 7. tur B'nin son oturumu
#    ~95 GB'la (DLCN'siz) bitti: sınır kesin değil, DLCN girdisinin önbelleği de payı olabilir. Önlem olarak:
#    DLCN/COHFACE hazırlama defterinde 30 s'ye kısaltılıp HF'ten gelir; izlemede DISK_LIMIT_GB'ta hazırlama durur
#    (B'nin biten oturumu ~92 GB'tı; sınır onun biraz üstünde, A/C aynı veriyi görsün).
def launch(name, cmd):
    return subprocess.Popen(["bash", "-c", cmd], cwd=REPO, env=env,
                            stdout=open(os.path.join(LOGS, name + ".log"), "a"), stderr=subprocess.STDOUT)

P = f"{PY} -u scripts/dl_prepare.py --out {DATA} --raw {RAW}"
jobs = {} if STD else {
    # D'de DLCN + COHFACE (~13 GB) da diskte: MCD 30 s'ye iner, toplam B'nin biten oturumunun (~92 GB) altında kalır
    "hazirla_mcd": launch("hazirla_mcd", f"{P} --mcd 600 --workers 2 --max-sec {30 if EK_VERI else 45}"),
    "hazirla_test_ubfc_pure": launch("hazirla_test_ubfc_pure",
        f"{P} --test-download --workers 2; "
        f"{P} --test-download --test-cameras IriunWebcam,USBVideo --test-sub test_kamera --no-test-ubfc --workers 2; "
        f"{P} --ubfc 34 --workers 1; {P} --pure 60 --workers 1"),
}
if STD:   # UBFC test kişileri (8) --test-download ile, kalan 34 --ubfc ile; PURE'un hepsi
    jobs["hazirla_test_ubfc_pure"] = launch("hazirla_test_ubfc_pure",
        f"{P} --test-download --workers 2; {P} --ubfc 34 --workers 1; {P} --pure 60 --workers 1")
print("veri hazırlama başladı")

In [ ]:
# 5) Eğitim: 300 video hazır olunca başlar; yeni gelenler ve hizalananlar her epoch başında eklenir
SUBS = ("mcd", "ubfc", "pure", "dlcn", "vipl", "cohface", "ubfcphys")
TESTS = ("test", "test_kamera", "test_vipl", "test_mmpd", "test_dlcn", "test_cohface", "test_ubfcphys")
def counts(subs=SUBS):
    return {s: len(glob.glob(os.path.join(DATA, s, "*.npz"))) for s in subs}

while sum(counts().values()) < (40 if STD else 300) and any(j.poll() is None for j in jobs.values()):
    print(time.strftime("%H:%M"), counts(), flush=True)
    time.sleep(120)

train = subprocess.Popen(
    [PY, "-u", "scripts/dl_train.py", "--data", DATA, "--out", OUT, *TRAIN_ARGS.split(), "--epochs", str(EPOCHS),
     "--steps", str(STEPS), "--deadline", TRAIN_UNTIL.strftime("%Y-%m-%d %H:%M")],
    cwd=REPO, env=env, stdout=open(os.path.join(LOGS, "egitim.log"), "a"), stderr=subprocess.STDOUT)
print("eğitim başladı:", " ".join(train.args[2:]))

In [ ]:
# 6) İzleme: 5 dakikada bir özet (gunlukler/izleme.log'a da yazılır ve yedeklenir), 30 dakikada bir yedek.
#    Bellek koruması: boş RAM 4 GB'ın altına inerse veri hazırlama duraklatılır, 8 GB'ı geçince sürer.
#    (6. tur ve 7. tur B hata dökümü olmadan durdu; Kaggle başarısız sürümün çıktısını saklamıyor.)
import signal
def tail(path, n):
    try:
        return open(path, encoding="utf-8", errors="replace").read().splitlines()[-n:]
    except OSError:
        return []

def tree_rss(pid):
    try:
        p = psutil.Process(pid)
        return p.memory_info().rss + sum(c.memory_info().rss for c in p.children(recursive=True))
    except Exception:
        return 0

def signal_tree(pid, sig):
    try:
        p = psutil.Process(pid)
        for q in [p] + p.children(recursive=True):
            q.send_signal(sig)
    except Exception:
        pass

def log_line(msg):
    print(msg, flush=True)
    with open(os.path.join(LOGS, "izleme.log"), "a", encoding="utf-8") as f:
        f.write(msg + "\n")

DISK_LIMIT_GB = 97 if KAGGLE else max(20, FREE0 - 10)   # Colab: başlangıçtaki boş alan - 10 GB pay
def used_gb():
    """Bu oturumun yazdığı veri (df değil: Kaggle'da df gerçek sınırı göstermiyor)."""
    tot = 0
    for root in (DATA, RAW, "/tmp/onceki", os.path.expanduser("~/.cache/huggingface")):
        for d, _, fs in os.walk(root):
            for f in fs:
                try:
                    tot += os.path.getsize(os.path.join(d, f))
                except OSError:
                    pass
    return tot / 1e9

paused, last_backup, disk_stop = False, time.time(), False
while train.poll() is None:
    try:
        vm = psutil.virtual_memory()
        avail = vm.available / 1e9
        live = {k: j for k, j in jobs.items() if j.poll() is None}
        if avail < 4 and live and not paused:
            for j in live.values():
                signal_tree(j.pid, signal.SIGSTOP)
            paused = True
            log_line(f"{time.strftime('%H:%M')} UYARI: boş RAM {avail:.1f} GB, veri hazırlama duraklatıldı")
        elif paused and avail > 8:
            for j in live.values():
                signal_tree(j.pid, signal.SIGCONT)
            paused = False
            log_line(f"{time.strftime('%H:%M')} boş RAM {avail:.1f} GB, veri hazırlama sürüyor")
        used = used_gb()
        if used > DISK_LIMIT_GB and live and not disk_stop:
            for j in live.values():
                signal_tree(j.pid, signal.SIGCONT)
                subprocess.run(["pkill", "-TERM", "-P", str(j.pid)])
                j.terminate()
            disk_stop = True
            log_line(f"{time.strftime('%H:%M')} DİSK SINIRI: oturum verisi {used:.0f} GB > {DISK_LIMIT_GB:.0f} GB, "
                     "veri hazırlama durduruldu (eğitim eldeki veriyle sürüyor)")
            live = {}
        mem = " ".join(f"{k.removeprefix('hazirla_')} {tree_rss(j.pid) / 1e9:.1f}" for k, j in live.items())
        gpu = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader"],  # defterde CUDA açılmasın
                             capture_output=True, text=True).stdout.strip().replace("\n", ", ") if shutil.which("nvidia-smi") else "yok"
        ep = [l for l in tail(os.path.join(LOGS, "egitim.log"), 400) if l.startswith("epoch ")]
        log_line(f"{time.strftime('%H:%M')} | {counts()} | test {counts(TESTS)}"
                 f" | veri {used:.0f}/{DISK_LIMIT_GB:.0f} GB, boş {shutil.disk_usage(WORK).free/1e9:.0f} GB"
                 f" | RAM boş {avail:.1f}/{vm.total/1e9:.0f} GB, eğitim {tree_rss(train.pid)/1e9:.1f} GB, {mem or 'hazırlama bitti'}"
                 f"{' (DURAKLATILDI)' if paused else ''} | GPU kullanılan {gpu}"
                 f" | {ep[-1] if ep else 'ilk epoch sürüyor'}")
    except Exception as e:
        print("izleme hatası:", repr(e), flush=True)
    if time.time() - last_backup > 1800:
        backup(f"eğitim sürüyor {time.strftime('%H:%M')}")
        last_backup = time.time()
    time.sleep(300)
if paused:
    for j in jobs.values():
        signal_tree(j.pid, signal.SIGCONT)
print("\n".join(tail(os.path.join(LOGS, "egitim.log"), 20)))
print("eğitim çıkış kodu:", train.returncode)
backup("eğitim bitti", wait=600)

In [ ]:
# 7) Değerlendirme: ana test (28), telefon/USB (test_kamera), VIPL (source1/3), MMPD, DLCN, COHFACE, UBFC-Phys
#    last.pth ile (7. turda son epoch en iyi epoch'tan iyiydi); best.pth karşılaştırmada ek satır
for j in jobs.values():
    try:
        if j.poll() is None:
            subprocess.run(["pkill", "-TERM", "-P", str(j.pid)])
            j.terminate()
    except Exception as e:
        print("durdurma hatası:", repr(e))
def hepsi(name, globs):
    """Standart protokolün test klasörü: hedef veri setinin bütün videoları (bağlantıyla, kopyasız)."""
    d = os.path.join(DATA, name)
    os.makedirs(d, exist_ok=True)
    for g in globs:
        for p in glob.glob(os.path.join(DATA, g)):
            q = os.path.join(d, os.path.basename(p))
            if not os.path.exists(q):
                os.symlink(p, q)
    return name

if STD:
    TESTS = (hepsi("test_ubfc_hepsi", ["ubfc/*.np[yz]", "test/ubfc_*.np[yz]"]) if DENEY == "S1" else
             hepsi("test_pure_hepsi", ["pure/*.np[yz]"]),)
print("test videoları:", counts(TESTS))
for t in TESTS:
    try:
        d = os.path.join(DATA, t)
        if not glob.glob(os.path.join(d, "*.npz")) or not os.path.exists(os.path.join(OUT, "last.pth")):
            continue
        out = OUT if t == "test" else os.path.join(OUT, t)
        run([PY, "scripts/dl_evaluate.py", "--model", os.path.join(OUT, "last.pth"), "--name", AD, "--test", d,
             "--data", DATA, "--out", out, *(["--prev", "yok.csv"] if STD else []), "--history", os.path.join(OUT, "gecmis.json"),
             "--compare", *([] if STD else [f"5. tur={TUR5}", f"3. tur (uygulamadaki)={TUR3}",
             f"7. tur B (MMRPhys sıfırdan)={TUR7B}", f"7. tur E son epoch (başlangıç)={TUR7E}"]),
             *([f"9. tur {DENEY} en iyi epoch={os.path.join(OUT, 'best.pth')}"]
               if os.path.exists(os.path.join(OUT, "best.pth")) else [])], f"degerlendirme_{t}")
    except Exception:
        import traceback; traceback.print_exc()
try:   # eğitime giren / çıkan videolar (kalite + hizalama)
    run([PY, "scripts/dl_quality.py", "--data", DATA, "--workers", "4", "--sources", *SUBS], "kalite")
    if os.path.exists(os.path.join(DATA, "kalite.csv")):
        shutil.copy(os.path.join(DATA, "kalite.csv"), OUT)
except Exception:
    import traceback; traceback.print_exc()
backup("değerlendirme bitti", wait=900)

In [ ]:
# 8) Çıktılar
for p in sorted(glob.glob(os.path.join(OUT, "**"), recursive=True)):
    if os.path.isfile(p):
        print(f"{os.path.getsize(p)/1e6:8.2f} MB  {p}")
os.chdir(WORK)
shutil.rmtree(REPO, ignore_errors=True)   # çıktıda yalnızca cikti/ kalsın
if not KAGGLE:   # Colab: soldaki Dosyalar panelinden indirilir (sonuçlar ayrıca Hugging Face'te)
    print("indirmek için:", shutil.make_archive(os.path.join(WORK, f"cikti_{RUN}"), "zip", OUT))
print("süre:", datetime.now() - T0)